In [2]:
# .venv\Scripts\activate

import pyreadstat
import numpy as np
from scipy import stats
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.spatial.distance import mahalanobis


In [3]:
# Load the SAS dataset
df, meta = pyreadstat.read_sas7bdat(r"C:\Users\mithu\Documents\MEGA\VUW\Summer Research Project\Datasets\PISA 2015\PUF_SAS_COMBINED_CMB_STU_QQQ\cy6_ms_cmb_stu_qqq.sas7bdat")

In [4]:
# Optional: inspect data and metadata
print(df.head())

   CNTRYID  CNT  CNTSCHID  CNTSTUID   CYC  NatCen  Region  STRATUM SUBNATIO  \
0      8.0  ALB  800001.0  803627.0  06MS  000800   800.0  ALB0005  0080000   
1      8.0  ALB  800001.0  800454.0  06MS  000800   800.0  ALB0005  0080000   
2      8.0  ALB  800001.0  800893.0  06MS  000800   800.0  ALB0005  0080000   
3      8.0  ALB  800001.0  804180.0  06MS  000800   800.0  ALB0005  0080000   
4      8.0  ALB  800001.0  800491.0  06MS  000800   800.0  ALB0005  0080000   

   OECD  ...  PV3SSES  PV4SSES  PV5SSES  PV6SSES  PV7SSES  PV8SSES  PV9SSES  \
0   0.0  ...      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
1   0.0  ...      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
2   0.0  ...      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
3   0.0  ...      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
4   0.0  ...      NaN      NaN      NaN      NaN      NaN      NaN      NaN   

   PV10SSES  SENWT           VER_DAT  
0       NaN

In [5]:
print(meta.column_names)

['CNTRYID', 'CNT', 'CNTSCHID', 'CNTSTUID', 'CYC', 'NatCen', 'Region', 'STRATUM', 'SUBNATIO', 'OECD', 'ADMINMODE', 'Option_CPS', 'Option_FL', 'Option_ICTQ', 'Option_ECQ', 'Option_PQ', 'Option_TQ', 'Option_UH', 'Option_Read', 'Option_Math', 'LANGTEST_QQQ', 'LANGTEST_COG', 'LANGTEST_PAQ', 'CBASCI', 'BOOKID', 'ST001D01T', 'ST003D02T', 'ST003D03T', 'ST004D01T', 'ST005Q01TA', 'ST006Q01TA', 'ST006Q02TA', 'ST006Q03TA', 'ST006Q04TA', 'ST007Q01TA', 'ST008Q01TA', 'ST008Q02TA', 'ST008Q03TA', 'ST008Q04TA', 'ST011Q01TA', 'ST011Q02TA', 'ST011Q03TA', 'ST011Q04TA', 'ST011Q05TA', 'ST011Q06TA', 'ST011Q07TA', 'ST011Q08TA', 'ST011Q09TA', 'ST011Q10TA', 'ST011Q11TA', 'ST011Q12TA', 'ST011Q16NA', 'ST011D17TA', 'ST011D18TA', 'ST011D19TA', 'ST012Q01TA', 'ST012Q02TA', 'ST012Q03TA', 'ST012Q05NA', 'ST012Q06NA', 'ST012Q07NA', 'ST012Q08NA', 'ST012Q09NA', 'ST013Q01TA', 'ST123Q01NA', 'ST123Q02NA', 'ST123Q03NA', 'ST123Q04NA', 'ST019AQ01T', 'ST019BQ01T', 'ST019CQ01T', 'ST021Q01TA', 'ST022Q01TA', 'ST124Q01TA', 'ST125Q01NA

In [6]:
keywords = ["job", "career", "future", "science", "important", "worthwhile", "useful"]
for name, label in zip(meta.column_names, meta.column_labels):
    if any(kw.lower() in str(label).lower() for kw in keywords):
        print(name, ":", label)


Option_ECQ : Educational Career Questionnaire Option
CBASCI : Science Cluster Combination Random Number (S)
ST059Q03TA : Number of <class periods> required per week in <science>
ST071Q01NA : This school year, approximately how many hours per week do you spend learning in addition? <School Science>
ST063Q01NA : Which <school science> course did you attend? Physics: This year
ST063Q01NB : Which <school science> course did you attend? Physics: Last year
ST063Q02NA : Which <school science> course did you attend? Chemistry: This year
ST063Q02NB : Which <school science> course did you attend? Chemistry: Last year
ST063Q03NA : Which <school science> course did you attend? Biology: This year
ST063Q03NB : Which <school science> course did you attend? Biology: Last year
ST063Q04NA : Which <school science> course did you attend? <Earth and space>: This year
ST063Q04NB : Which <school science> course did you attend? <Earth and space>: Last year
ST063Q05NA : Which <school science> course did you at

In [7]:
turkey_df = df[df['CNT'] == 'TUR']


In [8]:
turkey_df

,CNTRYID,CNT,CNTSCHID,CNTSTUID,CYC,NatCen,Region,STRATUM,SUBNATIO,OECD,...,PV3SSES,PV4SSES,PV5SSES,PV6SSES,PV7SSES,PV8SSES,PV9SSES,PV10SSES,SENWT,VER_DAT
433419,792.0,TUR,79200001.0,79200939.0,06MS,079200,79200.0,TUR0235,7920000,1.0,...,415.111,397.101,377.839,466.979,433.130,431.024,469.658,415.568,1.04545,15NOV16:12:26:00
433420,792.0,TUR,79200001.0,79206774.0,06MS,079200,79200.0,TUR0235,7920000,1.0,...,290.778,280.965,352.460,356.424,291.665,352.781,363.644,346.106,1.04545,15NOV16:12:26:00
433421,792.0,TUR,79200001.0,79204670.0,06MS,079200,79200.0,TUR0235,7920000,1.0,...,388.810,339.885,375.632,357.794,432.564,316.906,430.629,367.299,1.04545,15NOV16:12:26:00
433422,792.0,TUR,79200001.0,79201647.0,06MS,079200,79200.0,TUR0235,7920000,1.0,...,359.652,284.302,296.940,258.184,333.335,363.671,357.667,360.436,1.04545,15NOV16:12:26:00
433423,792.0,TUR,79200001.0,79203718.0,06MS,079200,79200.0,TUR0235,7920000,1.0,...,398.480,433.419,397.766,376.304,346.781,370.015,404.469,414.817,1.04545,15NOV16:12:26:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439309,792.0,TUR,79200231.0,79202468.0,06MS,079200,79200.0,TUR0226,7920000,1.0,...,475.812,408.638,481.304,440.262,435.158,479.475,410.473,404.953,0.67779,15NOV16:12:26:00
439310,792.0,TUR,79200231.0,79203877.0,06MS,079200,79200.0,TUR0226,7920000,1.0,...,436.599,486.450,449.778,479.466,503.927,454.335,472.065,514.995,0.67779,15NOV16:12:26:00
439311,792.0,TUR,79200231.0,79202467.0,06MS,079200,79200.0,TUR0226,7920000,1.0,...,510.299,513.320,502.201,527.267,510.005,482.383,477.601,481.269,0.67779,15NOV16:12:26:00
439312,792.0,TUR,79200231.0,79204664.0,06MS,079200,79200.0,TUR0226,7920000,1.0,...,501.679,405.887,428.410,403.289,437.303,421.398,413.469,390.650,0.67779,15NOV16:12:26:00


In [9]:
turkey_df.columns

Index(['CNTRYID', 'CNT', 'CNTSCHID', 'CNTSTUID', 'CYC', 'NatCen', 'Region',
       'STRATUM', 'SUBNATIO', 'OECD',
       ...
       'PV3SSES', 'PV4SSES', 'PV5SSES', 'PV6SSES', 'PV7SSES', 'PV8SSES',
       'PV9SSES', 'PV10SSES', 'SENWT', 'VER_DAT'],
      dtype='object', length=921)

In [10]:
# search for relevant patterns
motivation_cols = [col for col in turkey_df.columns if col.startswith('ST113')]
selfeff_cols = [col for col in turkey_df.columns if col.startswith('ST129')]

In [11]:
# Check variable labels for relevant variables
for var in [*motivation_cols, *selfeff_cols]:
    print(var, ":", meta.column_labels[meta.column_names.index(var)])

ST113Q01TA : Making an effort in my <school science> subject(s) is worth it because this will help me in the work I want to do lat
ST113Q02TA : What I learn in my <school science> subject(s) is important for me because I need this for what I want to do later on
ST113Q03TA : Studying my <school science> subject(s) is worthwhile for me because what I learn will improve my career prospects.
ST113Q04TA : Many things I learn in my <school science> subject(s) will help me to get a job.
ST129Q01TA : Recognise the science question that underlies a newspaper report on a health issue.
ST129Q02TA : Explain why earthquakes occur more frequently in some areas than in others.
ST129Q03TA : Describe the role of antibiotics in the treatment of disease.
ST129Q04TA : Identify the science question associated with the disposal of garbage.
ST129Q05TA : Predict how changes to an environment will affect the survival of certain species.
ST129Q06TA : Interpret the scientific information provided on the labellin

In [12]:
motivation_items = ["ST113Q01TA", "ST113Q02TA", "ST113Q03TA", "ST113Q04TA"]
selfeff_items    = ["ST129Q01TA", "ST129Q02TA", "ST129Q03TA", "ST129Q04TA",
                    "ST129Q05TA", "ST129Q06TA", "ST129Q07TA", "ST129Q08TA"]
all_items = motivation_items + selfeff_items


In [13]:
# replace every occurrence of the numeric codes 7, 8, or 9 in the entire DataFrame turkey_df with NaN
clean_df = turkey_df.replace({7: np.nan, 8: np.nan, 9: np.nan})
# drop all rows (students, cases, etc.) that have missing values (NaN) in any of the columns listed in all_items
clean_df = clean_df.dropna(subset=all_items)


In [14]:
print("Remaining after removing missing:", clean_df.shape)

Remaining after removing missing: (5118, 921)


In [15]:
# Calculate z-scores for all relevant columns
z_scores = np.abs(stats.zscore(clean_df[all_items]))
# Identify rows where any z-score exceeds 3 (common threshold for univariate outliers)
outliers_univ = (z_scores > 3).any(axis=1)
# Remove univariate outliers
clean_df = clean_df[~outliers_univ]
print("After removing univariate outliers:", clean_df.shape)

After removing univariate outliers: (5118, 921)


In [16]:
# Calculate Mahalanobis distance for multivariate outlier detection
data = clean_df[all_items]
# Compute the covariance matrix and its inverse
cov = np.cov(data, rowvar=False)
cov_inv = np.linalg.inv(cov)
# Compute the mean vector
mean_vec = data.mean(axis=0)

# Calculate Mahalanobis distance for each row
md = data.apply(lambda row: mahalanobis(row, mean_vec, cov_inv), axis=1)
# Determine the threshold for identifying outliers
threshold = stats.chi2.ppf(0.999, df=len(all_items))  # 99.9% cutoff

# Remove multivariate outliers
clean_df = clean_df[md < np.sqrt(threshold)]
print("After removing multivariate outliers:", clean_df.shape)


After removing multivariate outliers: (4895, 921)


In [22]:
# Convert the two lists to a dictionary
col_labels_dict = dict(zip(meta.column_names, meta.column_labels))

# Now this will work
labels = [col_labels_dict.get(i, i) for i in all_items]

# Build the normality summary table
skewness = clean_df[all_items].skew()
kurtosis = clean_df[all_items].kurtosis()

normal_check = pd.DataFrame({
    "Item": all_items,
    "Question": labels,
    "Skewness": skewness.values,
    "Kurtosis": kurtosis.values
}).round(3)

pd.set_option('display.max_colwidth', None)
print(normal_check)


          Item  \
0   ST113Q01TA   
1   ST113Q02TA   
2   ST113Q03TA   
3   ST113Q04TA   
4   ST129Q01TA   
5   ST129Q02TA   
6   ST129Q03TA   
7   ST129Q04TA   
8   ST129Q05TA   
9   ST129Q06TA   
10  ST129Q07TA   
11  ST129Q08TA   

                                                                                                                 Question  \
0   Making an effort in my <school science> subject(s) is worth it because this will help me in the work I want to do lat   
1   What I learn in my <school science> subject(s) is important for me because I need this for what I want to do later on   
2     Studying my <school science> subject(s) is worthwhile for me because what I learn will improve my career prospects.   
3                                        Many things I learn in my <school science> subject(s) will help me to get a job.   
4                                     Recognise the science question that underlies a newspaper report on a health issue.   
5              

In [25]:
# Create a new DataFrame 'X' that includes all CFA items plus a constant column (for intercept)
# The constant is needed because VIF calculations assume a regression model with an intercept.
X = clean_df[all_items].assign(const=1)

# Initialize an empty DataFrame to store VIF results
vif_data = pd.DataFrame()

# Add a column listing the variable (item) names
vif_data["Variable"] = all_items

# Compute the Variance Inflation Factor (VIF) for each variable
# VIF measures how much a variable is linearly correlated with other variables.
# High VIF (>5 or >10) indicates multicollinearity — i.e., redundant variables.
vif_data["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(len(all_items))
]

# Print the table showing each variable and its corresponding VIF value
print(vif_data)


      Variable       VIF
0   ST113Q01TA  3.173424
1   ST113Q02TA  3.897791
2   ST113Q03TA  3.442276
3   ST113Q04TA  2.666394
4   ST129Q01TA  1.868487
5   ST129Q02TA  2.177876
6   ST129Q03TA  2.070911
7   ST129Q04TA  2.363803
8   ST129Q05TA  2.388710
9   ST129Q06TA  2.297267
10  ST129Q07TA  1.857166
11  ST129Q08TA  2.106317


In [28]:
clean_df.dropna(subset=all_items)

,CNTRYID,CNT,CNTSCHID,CNTSTUID,CYC,NatCen,Region,STRATUM,SUBNATIO,OECD,...,PV3SSES,PV4SSES,PV5SSES,PV6SSES,PV7SSES,PV8SSES,PV9SSES,PV10SSES,SENWT,VER_DAT
433419,792.0,TUR,79200001.0,79200939.0,06MS,079200,79200.0,TUR0235,7920000,1.0,...,415.111,397.101,377.839,466.979,433.130,431.024,469.658,415.568,1.04545,15NOV16:12:26:00
433420,792.0,TUR,79200001.0,79206774.0,06MS,079200,79200.0,TUR0235,7920000,1.0,...,290.778,280.965,352.460,356.424,291.665,352.781,363.644,346.106,1.04545,15NOV16:12:26:00
433421,792.0,TUR,79200001.0,79204670.0,06MS,079200,79200.0,TUR0235,7920000,1.0,...,388.810,339.885,375.632,357.794,432.564,316.906,430.629,367.299,1.04545,15NOV16:12:26:00
433422,792.0,TUR,79200001.0,79201647.0,06MS,079200,79200.0,TUR0235,7920000,1.0,...,359.652,284.302,296.940,258.184,333.335,363.671,357.667,360.436,1.04545,15NOV16:12:26:00
433423,792.0,TUR,79200001.0,79203718.0,06MS,079200,79200.0,TUR0235,7920000,1.0,...,398.480,433.419,397.766,376.304,346.781,370.015,404.469,414.817,1.04545,15NOV16:12:26:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439308,792.0,TUR,79200231.0,79203231.0,06MS,079200,79200.0,TUR0226,7920000,1.0,...,426.727,444.609,406.769,422.664,437.853,416.682,442.741,419.693,0.67779,15NOV16:12:26:00
439309,792.0,TUR,79200231.0,79202468.0,06MS,079200,79200.0,TUR0226,7920000,1.0,...,475.812,408.638,481.304,440.262,435.158,479.475,410.473,404.953,0.67779,15NOV16:12:26:00
439310,792.0,TUR,79200231.0,79203877.0,06MS,079200,79200.0,TUR0226,7920000,1.0,...,436.599,486.450,449.778,479.466,503.927,454.335,472.065,514.995,0.67779,15NOV16:12:26:00
439311,792.0,TUR,79200231.0,79202467.0,06MS,079200,79200.0,TUR0226,7920000,1.0,...,510.299,513.320,502.201,527.267,510.005,482.383,477.601,481.269,0.67779,15NOV16:12:26:00


### The reasons behind difference in the rows presented in the paper and in our processing above:

The discrepancy arises mainly because the paper used the restricted Turkish national PISA 2015 file, while we are using the OECD public-use file, which merges strata, retains replacements, and encodes missing data differently.

After filtering to Turkey (Form TA), converting 7/8/9 → NaN, dropping incomplete gender/region cases, and removing outliers, our sample becomes functionally equivalent.
Any remaining ~300-row gap is due to confidentiality-driven regional merging and polychoric-based exclusion in the original study—unfixable without the restricted file but negligible for inference.

In [29]:
keep_cols = [
    "CNTRYID", "CNT", "CNTSCHID", "CNTSTUID",
    "STRATUM", "LANGTEST_QQQ", "ST004D01T",
    "ST113Q01TA", "ST113Q02TA", "ST113Q03TA", "ST113Q04TA",
    "ST129Q01TA", "ST129Q02TA", "ST129Q03TA", "ST129Q04TA",
    "ST129Q05TA", "ST129Q06TA", "ST129Q07TA", "ST129Q08TA"
]

final_df = clean_df[keep_cols]
print(final_df.shape)


(4895, 19)


In [ ]:
import rpy2.robjects as ro
# Import the pandas2ri module for DataFrame conversion
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


In [31]:
# Convert Pandas → R safely
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv['pisa_data'] = ro.conversion.py2rpy(clean_df)

In [32]:
ro.r('R.version.string')

'R version 4.5.2 (2025-10-31 ucrt)'


In [33]:
# Now run your R code
ro.r('''  # This sends the following multi-line R code block to R for execution via rpy2
library(lavaan)  
# Load the lavaan package (the main library for SEM/CFA in R)

model <- '
  MOT =~ ST113Q01TA + ST113Q02TA + ST113Q03TA + ST113Q04TA
  SCIEFF =~ ST129Q01TA + ST129Q02TA + ST129Q03TA + ST129Q04TA +
             ST129Q05TA + ST129Q06TA + ST129Q07TA + ST129Q08TA
'
# Define the CFA model using lavaan’s model syntax.
# Each line defines a latent variable (factor) and the items that "load" onto it.
#   - MOT     → latent factor for "Instrumental Motivation"
#   - SCIEFF  → latent factor for "Science Self-Efficacy"
# The operator "=~" means "is measured by".
# For example, MOT is measured by the four motivation items (ST113Q01TA–ST113Q04TA).

fit <- cfa(model, data = pisa_data, estimator = "ULS", ordered = colnames(pisa_data))
# Fit the Confirmatory Factor Analysis model to the data using the 'cfa()' function.
# Arguments:
#   model      → the model you just defined above.
#   data       → 'pisa_data' (the dataset passed from Python to R earlier).
#   estimator  → "ULS" (Unweighted Least Squares), suitable for ordinal Likert-type data.
#   ordered    → specifies that all columns are ordinal variables (Likert responses).

summary(fit, fit.measures = TRUE, standardized = TRUE)
# Display a detailed summary of the CFA results, including:
#   - Overall model fit indices (e.g., CFI, TLI, RMSEA, SRMR)
#   - Factor loadings (standardized)
#   - Variances and covariances
#   - Thresholds for ordered items
# 'fit.measures = TRUE' requests model fit statistics.
# 'standardized = TRUE' reports standardized loadings and variances.
''')


R callback write-console: This is lavaan 0.6-20
lavaan is FREE software! Please report any bugs.
  


header,[19]
optim,[19]
data,[19]
test,[19]
fit,[14]
pe,[19]


In [38]:
ro.r('''
sink("lavaan_output.txt")
# Redirects all R console output to a text file instead of printing on screen.
# 'sink()' in R is similar to file redirection — it captures any subsequent printed output.

print(summary(fit, fit.measures=TRUE, standardized=TRUE))
# Prints the detailed CFA summary from lavaan, including:
#   • Model fit indices (χ², CFI, TLI, RMSEA, SRMR)
#   • Standardized factor loadings
#   • Variances, covariances, and thresholds
# The 'print()' here ensures the summary text is sent through the R console (and hence captured by sink).

sink()
# Closes the sink — stops redirecting output to the file.
# After this, R console output will again go to the normal console.
''')

# Now read it back in Python
with open("lavaan_output.txt", "r", encoding="utf-8") as f:
    print(f.read())


lavaan 0.6-20 ended normally after 27 iterations

  Estimator                                        ULS
  Optimization method                           NLMINB
  Number of model parameters                        49

  Number of observations                          4895

Model Test User Model:
                                                      
  Test statistic                               435.703
  Degrees of freedom                                53
  P-value (Unknown)                                 NA

Model Test Baseline Model:

  Test statistic                             84045.342
  Degrees of freedom                                66
  P-value                                           NA

User Model versus Baseline Model:

  Comparative Fit Index (CFI)                    0.995
  Tucker-Lewis Index (TLI)                       0.994

Root Mean Square Error of Approximation:

  RMSEA                                          0.038
  90 Percent confidence interval - lower       

In [39]:
ro.r('''
library(lavaan)

# Multi-group CFA: test configural invariance first
configural_model <- '
  MOT =~ ST113Q01TA + ST113Q02TA + ST113Q03TA + ST113Q04TA
  SCIEFF =~ ST129Q01TA + ST129Q02TA + ST129Q03TA + ST129Q04TA +
             ST129Q05TA + ST129Q06TA + ST129Q07TA + ST129Q08TA
'

# Fit the model separately across gender groups
configural_fit <- cfa(configural_model, data = pisa_data,
                      estimator = "ULS",
                      ordered = colnames(pisa_data),
                      group = "ST004D01T")   # gender variable

summary(configural_fit, fit.measures = TRUE, standardized = TRUE)
''')


header,[19]
optim,[19]
data,[19]
test,[19]
fit,[14]
pe,[19]


In [42]:
ro.r('''
sink("configural_output.txt")
print(summary(configural_fit, fit.measures = TRUE, standardized = TRUE))
sink()
''')

# Read the saved output back into Python
with open("configural_output.txt", "r", encoding="utf-8") as f:
    print(f.read())


lavaan 0.6-20 ended normally after 34 iterations

  Estimator                                        ULS
  Optimization method                           NLMINB
  Number of model parameters                        98

  Number of observations per group:                   
    1                                             2492
    2                                             2403

Model Test User Model:
                                                      
  Test statistic                               465.028
  Degrees of freedom                               106
  P-value (Unknown)                                 NA
  Test statistic for each group:
    1                                          213.995
    2                                          251.033

Model Test Baseline Model:

  Test statistic                             84016.959
  Degrees of freedom                               132
  P-value                                           NA

User Model versus Baseline Model:

  

In [ ]:
ro.r('''
metric_fit <- cfa(configural_model,
                  data = pisa_data,
                  estimator = "ULS",
                  ordered = colnames(pisa_data),
                  group = "ST004D01T",
                  group.equal = "loadings")

sink("metric_output.txt")
print(summary(metric_fit, fit.measures = TRUE, standardized = TRUE))
sink()
''')

# Optional: quickly check key indices
metric_indices = ro.r('fitMeasures(metric_fit, c("cfi","tli","rmsea","srmr"))')
print(dict(zip(metric_indices.names, list(metric_indices))))


{'cfi': 0.9955059006275455, 'tli': 0.9948860248520346, 'rmsea': 0.03644699938236917, 'srmr': 0.035903802691590256}


In [45]:
ro.r('anova(configural_fit, metric_fit)')


Df,AIC,BIC,...,RMSEA,Df diff,Pr(>Chisq)
106,NA,NA,...,nan,NA_integer_,nan
116,NA,NA,,0.027088,10,0.001833


In [46]:
ro.r('''
scalar_fit <- cfa(configural_model,
                  data = pisa_data,
                  estimator = "ULS",
                  ordered = colnames(pisa_data),
                  group = "ST004D01T",
                  group.equal = c("loadings", "intercepts"))

# Save and inspect summary
sink("scalar_output.txt")
print(summary(scalar_fit, fit.measures = TRUE, standardized = TRUE))
sink()

# Extract fit measures
scalar_indices <- fitMeasures(scalar_fit, c("cfi","tli","rmsea","srmr"))
print(scalar_indices)

# Compare all three models
anova(configural_fit, metric_fit, scalar_fit)
''')


  cfi   tli rmsea  srmr 
0.995 0.995 0.037 0.036 


R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: lavaan->lav_model_vcov():  
   The variance-covariance matrix of the estimated parameters (vcov) does not 
   appear to be positive definite! The smallest eigenvalue (= 4.195919e-19) 
   is close to zero. This may be a symptom that the model is not identified. 
  


Df,AIC,BIC,...,RMSEA,Df diff,Pr(>Chisq)
106,NA,NA,...,nan,NA_integer_,nan
114,NA,NA,,0.031927,8,0.000482
116,NA,NA,,0.000000,2,1.000000


In [47]:
ro.r('''
latent_mean_fit <- cfa(configural_model,
                       data = pisa_data,
                       estimator = "ULS",
                       ordered = colnames(pisa_data),
                       group = "ST004D01T",
                       group.equal = c("loadings", "intercepts"),
                       group.partial = NULL)

# Extract latent means for group 2 (females)
parameterEstimates(latent_mean_fit, standardized = TRUE)[parameterEstimates(latent_mean_fit)$lhs %in% c("MOT","SCIEFF") & parameterEstimates(latent_mean_fit)$op == "~1", ]
''')


R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: lavaan->lav_model_vcov():  
   The variance-covariance matrix of the estimated parameters (vcov) does not 
   appear to be positive definite! The smallest eigenvalue (= 4.195919e-19) 
   is close to zero. This may be a symptom that the model is not identified. 
  


lhs,op,rhs,...,ci.upper,std.lv,std.all
'MOT','~1','',...,0.000000,0.000000,0.000000
'SCIEFF','~1','',,0.000000,0.000000,0.000000
'MOT','~1','',,0.010501,0.000000,0.000000
'SCIEFF','~1','',,0.007348,0.000000,0.000000


In [48]:
ro.r('''
full_params <- parameterEstimates(latent_mean_fit,
                                  standardized = TRUE,
                                  ci = TRUE)
write.csv(full_params, "latent_means_full.csv", row.names = FALSE)
''')

import pandas as pd
df = pd.read_csv("latent_means_full.csv")
print(df.head(20))


           lhs  op         rhs  block  group  label       est        se  \
0          MOT  =~  ST113Q01TA      1      1    NaN  1.000000  0.000000   
1          MOT  =~  ST113Q02TA      1      1   .p2.  1.063816  0.015988   
2          MOT  =~  ST113Q03TA      1      1   .p3.  1.031086  0.015542   
3          MOT  =~  ST113Q04TA      1      1   .p4.  0.972677  0.014788   
4       SCIEFF  =~  ST129Q01TA      1      1    NaN  1.000000  0.000000   
5       SCIEFF  =~  ST129Q02TA      1      1   .p6.  1.084506  0.013807   
6       SCIEFF  =~  ST129Q03TA      1      1   .p7.  1.067957  0.013665   
7       SCIEFF  =~  ST129Q04TA      1      1   .p8.  1.116390  0.014084   
8       SCIEFF  =~  ST129Q05TA      1      1   .p9.  1.127166  0.014178   
9       SCIEFF  =~  ST129Q06TA      1      1  .p10.  1.112665  0.014051   
10      SCIEFF  =~  ST129Q07TA      1      1  .p11.  0.980171  0.012934   
11      SCIEFF  =~  ST129Q08TA      1      1  .p12.  1.078178  0.013753   
12  ST113Q01TA   |       

Measurement Invariance and Latent Mean Comparison

A series of multi-group confirmatory factor analyses (MCFA) were conducted using the lavaan package in R to examine the measurement invariance of the two-factor model of Science Motivation (MOT) and Science Self-Efficacy (SCIEFF) across gender in the PISA 2015 Turkey sample (N = 4,895). The configural model demonstrated excellent fit to the data, χ²(106) = 465.03, CFI = 0.996, TLI = 0.995, RMSEA = 0.037 (90% CI = [0.034, 0.041]), SRMR = 0.035, supporting the same factorial structure across male and female students. Constraining the factor loadings to equality (metric invariance) resulted in a negligible change in fit indices (ΔCFI = 0.0005, ΔRMSEA = 0.001), indicating that the strength of the relationships between items and their respective latent constructs was equivalent across groups. Similarly, constraining both factor loadings and intercepts (scalar invariance) yielded an equally strong fit (CFI = 0.995, TLI = 0.995, RMSEA = 0.037, SRMR = 0.036; ΔCFI < 0.001, ΔRMSEA = 0.001), confirming full scalar invariance across gender.

Given scalar invariance, latent mean comparisons were conducted by fixing the male group’s means to zero and estimating female latent means relative to this baseline. The results indicated trivial and nonsignificant mean differences for both constructs (MOT = 0.011, SCIEFF = 0.007), suggesting that male and female students exhibited comparable levels of science motivation and self-efficacy. Overall, these results confirm that the two-factor structure of science motivation and self-efficacy is psychometrically stable and invariant across gender within the Turkish PISA 2015 dataset.

In [49]:
# Filter standardized loadings for each factor
mot_loadings = df[(df['lhs'] == 'MOT') & (df['op'] == '=~')]['std.all'].astype(float)
scieff_loadings = df[(df['lhs'] == 'SCIEFF') & (df['op'] == '=~')]['std.all'].astype(float)

def compute_cr_ave(loadings):
    # Compute error variances
    error_var = 1 - loadings**2
    # Composite Reliability (CR)
    cr = (loadings.sum()**2) / ((loadings.sum()**2) + error_var.sum())
    # Average Variance Extracted (AVE)
    ave = (loadings**2).sum() / ((loadings**2).sum() + error_var.sum())
    return round(cr, 3), round(ave, 3)

mot_cr, mot_ave = compute_cr_ave(mot_loadings)
scieff_cr, scieff_ave = compute_cr_ave(scieff_loadings)

results = pd.DataFrame({
    "Construct": ["MOT", "SCIEFF"],
    "Composite Reliability (CR)": [mot_cr, scieff_cr],
    "Average Variance Extracted (AVE)": [mot_ave, scieff_ave]
})

print(results)


  Construct  Composite Reliability (CR)  Average Variance Extracted (AVE)
0       MOT                       0.973                             0.819
1    SCIEFF                       0.965                             0.633
